In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [2]:
import json
import pandas as pd
from pprint import pprint
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import numpy as np

Creating group_techniques df for previous time periods

In [3]:
import json 
import pandas as pd
def create_df(file_name):
    with open(file_name, "r", encoding="utf-8") as f:
        stix = json.load(f)
    objects = stix.get("objects", [])
    group_objs = [o for o in objects if o.get("type") == "intrusion-set"]
    rel_objs = [o for o in objects if o.get("type") == "relationship"]
    tech_objs = [o for o in objects if o.get("type") == "attack-pattern"]

    def technique_row(o):
        # external id e.g. T1003 usually in external_references where source_name == 'mitre-attack'
        ext_refs = o.get("external_references", [])
        mitre_ref = next((r for r in ext_refs if r.get("source_name") == "mitre-attack"), {})
        external_id = mitre_ref.get("external_id")
        # kill_chain_phases may contain tactic phase names
        kcp = o.get("kill_chain_phases") or o.get("kill_chain_phases", []) or []
        phases = [p.get("phase_name") for p in kcp if isinstance(p, dict) and p.get("phase_name")]
        platforms = o.get("x_mitre_platforms") or o.get("x-mitre-platforms") or []
        data_sources = o.get("x_mitre_data_sources") or []
        return {
            "id": o.get("id"),
            "tech_name": o.get("name"),
            "external_id": external_id,
            "description": o.get("description"),
            "platforms": platforms,
            "kill_chain_phases": phases,
            "raw": o
        }

    tech_df = pd.DataFrame([technique_row(o) for o in tech_objs])
    # Filter relevant "uses" relationships
    uses_rels = [
        r for r in rel_objs
        if r.get("relationship_type") == "uses"
        and r.get("source_ref", "").startswith("intrusion-set--")
        and r.get("target_ref", "").startswith("attack-pattern--")
    ]

    rows = []
    for r in uses_rels:
        group_id = r["source_ref"]
        tech_id = r["target_ref"]

        # Lookup group name
        group_name = next((g["name"] for g in group_objs if g["id"] == group_id), None)

        # Lookup technique information
        tech = next((t for t in tech_df.to_dict("records") if t["id"] == tech_id), None)

        rows.append({
            "group_id": group_id,
            "group_name": group_name,
            "technique_id": tech_id,
            "technique_name": tech.get("tech_name") if tech else None,
            "tactic": tech.get("kill_chain_phases") if tech else None,
            "technique_description": tech.get("description") if tech else None,
            "relationship_description": r.get("description"),
        })
    ret = pd.DataFrame(rows)
    time_period = stix.get("objects")[0].get("modified")
    return time_period, ret

In [4]:
dfs = dict() 
# group_techniques_df is already the latest time period
dfs[1] = group_techniques_df # "2025-11-13T14:00:00.188Z"
# 1 time period back
time_period, df = create_df("../../enterprise-attack/enterprise-attack-17.1.json")
print(f"time period 2 is {time_period} ")
dfs[2] = df
# 2 time periods back
time_period, df = create_df("../../enterprise-attack/enterprise-attack-16.1.json")
print(f"time period 3 is {time_period} ")
dfs[3] = df
time_period, df = create_df("../../enterprise-attack/enterprise-attack-15.1.json")
print(f"time period 4 is {time_period} ")
dfs[4] = df
time_period, df = create_df("../../enterprise-attack/enterprise-attack-14.1.json")
print(f"time period 5 is {time_period} ")
dfs[5] = df
time_period, df = create_df("../../enterprise-attack/enterprise-attack-13.1.json")
print(f"time period 6 is {time_period} ")
dfs[6] = df
time_period, df = create_df("../../enterprise-attack/enterprise-attack-12.1.json")
print(f"time period 7 is {time_period} ")
dfs[7] = df

time period 2 is 2025-05-06T14:00:00.188Z 
time period 3 is 2024-11-12T14:00:00.188Z 
time period 4 is 2024-05-02T14:00:00.188Z 
time period 5 is 2023-11-14T14:00:00.188Z 
time period 6 is 2023-05-09T14:00:00.188Z 
time period 7 is 2022-11-08T14:00:00.188Z 


In [5]:
for time_period, df in dfs.items():
    print(f"Time Period: {time_period}, Number of Records: {len(df)}")
    display(df.head(10))

Time Period: 1, Number of Records: 4362


,group_id,group_name,technique_id,technique_name,tactic,technique_description,relationship_description
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,Indrik Spider,attack-pattern--65f2d882-3f41-4d48-8a06-29af77...,LSASS Memory,[credential-access],Adversaries may attempt to access credential m...,[Indrik Spider](https://attack.mitre.org/group...
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa...,LuminousMoth,attack-pattern--10ffac09-e42d-4f56-ab20-db94c6...,Steal Web Session Cookie,[credential-access],An adversary may steal web application or serv...,[LuminousMoth](https://attack.mitre.org/groups...
2,intrusion-set--918da025-04bd-48af-b6c4-f3e4d1b...,Medusa Group,attack-pattern--f5d8eed6-48a9-4cdf-a3d7-d1ffa9...,Inhibit System Recovery,[impact],Adversaries may delete or remove built-in data...,[Medusa Group](https://attack.mitre.org/groups...
3,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,attack-pattern--635cbe30-392d-4e27-978e-667743...,Local Account,[persistence],Adversaries may create a local account to main...,[Wizard Spider](https://attack.mitre.org/group...
4,intrusion-set--3753cc21-2dae-4dfb-8481-d004e74...,FIN7,attack-pattern--ef67e13e-5598-4adc-bdb2-998225...,Malicious Link,[execution],An adversary may rely upon a user clicking a m...,[FIN7](https://attack.mitre.org/groups/G0046) ...
5,intrusion-set--461b8e25-8f4a-4ea2-a4a8-e39df7c...,UNC3886,attack-pattern--b0533c6e-8fea-4788-874f-b799ca...,Indicator Removal from Tools,[defense-evasion],Adversaries may remove indicators from tools i...,[UNC3886](https://attack.mitre.org/groups/G104...
6,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--970a3432-3237-47ad-bcca-7d8cbb...,PowerShell,[execution],Adversaries may abuse PowerShell commands and ...,[WIRTE](https://attack.mitre.org/groups/G0090)...
7,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--b18eae87-b469-4e14-b454-b171b4...,Non-Standard Port,[command-and-control],Adversaries may communicate using a protocol a...,[WIRTE](https://attack.mitre.org/groups/G0090)...
8,intrusion-set--461b8e25-8f4a-4ea2-a4a8-e39df7c...,UNC3886,attack-pattern--63b24abc-5702-4745-b1e4-ac70b2...,Search Threat Vendor Data,[reconnaissance],Threat actors may seek information/indicators ...,[UNC3886](https://attack.mitre.org/groups/G104...
9,intrusion-set--1c63d4ec-0a75-4daa-b1df-0d11af3...,Dragonfly,attack-pattern--53ac20cd-aca3-406e-9aa0-9fc7fd...,Archive Collected Data,[collection],An adversary may compress and/or encrypt data ...,[Dragonfly](https://attack.mitre.org/groups/G0...


Time Period: 2, Number of Records: 4046


,group_id,group_name,technique_id,technique_name,tactic,technique_description,relationship_description
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,Indrik Spider,attack-pattern--65f2d882-3f41-4d48-8a06-29af77...,LSASS Memory,[credential-access],Adversaries may attempt to access credential m...,[Indrik Spider](https://attack.mitre.org/group...
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa...,LuminousMoth,attack-pattern--10ffac09-e42d-4f56-ab20-db94c6...,Steal Web Session Cookie,[credential-access],An adversary may steal web application or serv...,[LuminousMoth](https://attack.mitre.org/groups...
2,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,attack-pattern--635cbe30-392d-4e27-978e-667743...,Local Account,[persistence],Adversaries may create a local account to main...,[Wizard Spider](https://attack.mitre.org/group...
3,intrusion-set--3753cc21-2dae-4dfb-8481-d004e74...,FIN7,attack-pattern--ef67e13e-5598-4adc-bdb2-998225...,Malicious Link,[execution],An adversary may rely upon a user clicking a m...,[FIN7](https://attack.mitre.org/groups/G0046) ...
4,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--970a3432-3237-47ad-bcca-7d8cbb...,PowerShell,[execution],Adversaries may abuse PowerShell commands and ...,[WIRTE](https://attack.mitre.org/groups/G0090)...
5,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--b18eae87-b469-4e14-b454-b171b4...,Non-Standard Port,[command-and-control],Adversaries may communicate using a protocol a...,[WIRTE](https://attack.mitre.org/groups/G0090)...
6,intrusion-set--1c63d4ec-0a75-4daa-b1df-0d11af3...,Dragonfly,attack-pattern--53ac20cd-aca3-406e-9aa0-9fc7fd...,Archive Collected Data,[collection],An adversary may compress and/or encrypt data ...,[Dragonfly](https://attack.mitre.org/groups/G0...
7,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,attack-pattern--e7cbc1de-1f79-48ee-abfd-da1241...,Code Signing Certificates,[resource-development],Adversaries may buy and/or steal code signing ...,[Wizard Spider](https://attack.mitre.org/group...
8,intrusion-set--96e239be-ad99-49eb-b127-3007b8c...,Equation,attack-pattern--dfebc3b7-d19d-450b-81c7-6dafe4...,Hidden File System,[defense-evasion],Adversaries may use a hidden file system to co...,[Equation](https://attack.mitre.org/groups/G00...
9,intrusion-set--4ca1929c-7d64-4aab-b849-badbfc0...,OilRig,attack-pattern--e7cbc1de-1f79-48ee-abfd-da1241...,Code Signing Certificates,[resource-development],Adversaries may buy and/or steal code signing ...,[OilRig](https://attack.mitre.org/groups/G0049...


Time Period: 3, Number of Records: 4323


...

,group_id,group_name,technique_id,technique_name,tactic,technique_description,relationship_description
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,Indrik Spider,attack-pattern--65f2d882-3f41-4d48-8a06-29af77...,LSASS Memory,[credential-access],Adversaries may attempt to access credential m...,[Indrik Spider](https://attack.mitre.org/group...
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa...,LuminousMoth,attack-pattern--10ffac09-e42d-4f56-ab20-db94c6...,Steal Web Session Cookie,[credential-access],An adversary may steal web application or serv...,[LuminousMoth](https://attack.mitre.org/groups...
2,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,attack-pattern--635cbe30-392d-4e27-978e-667743...,Local Account,[persistence],Adversaries may create a local account to main...,[Wizard Spider](https://attack.mitre.org/group...
3,intrusion-set--3753cc21-2dae-4dfb-8481-d004e74...,FIN7,attack-pattern--ef67e13e-5598-4adc-bdb2-998225...,Malicious Link,[execution],An adversary may rely upon a user clicking a m...,[FIN7](https://attack.mitre.org/groups/G0046) ...
4,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--970a3432-3237-47ad-bcca-7d8cbb...,PowerShell,[execution],Adversaries may abuse PowerShell commands and ...,[WIRTE](https://attack.mitre.org/groups/G0090)...
5,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--b18eae87-b469-4e14-b454-b171b4...,Non-Standard Port,[command-and-control],Adversaries may communicate using a protocol a...,[WIRTE](https://attack.mitre.org/groups/G0090)...
6,intrusion-set--1c63d4ec-0a75-4daa-b1df-0d11af3...,Dragonfly,attack-pattern--53ac20cd-aca3-406e-9aa0-9fc7fd...,Archive Collected Data,[collection],An adversary may compress and/or encrypt data ...,[Dragonfly](https://attack.mitre.org/groups/G0...
7,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,attack-pattern--e7cbc1de-1f79-48ee-abfd-da1241...,Code Signing Certificates,[resource-development],Adversaries may buy and/or steal code signing ...,[Wizard Spider](https://attack.mitre.org/group...
8,intrusion-set--96e239be-ad99-49eb-b127-3007b8c...,Equation,attack-pattern--dfebc3b7-d19d-450b-81c7-6dafe4...,Hidden File System,[defense-evasion],Adversaries may use a hidden file system to co...,[Equation](https://attack.mitre.org/groups/G00...
9,intrusion-set--c21dd6f1-1364-4a70-a1f7-783080e...,Fox Kitten,attack-pattern--e6919abc-99f9-4c6c-95a5-14761e...,Ingress Tool Transfer,[command-and-control],Adversaries may transfer tools or other files ...,[Fox Kitten](https://attack.mitre.org/groups/G...


Time Period: 6, Number of Records: 3562


,group_id,group_name,technique_id,technique_name,tactic,technique_description,relationship_description
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,Indrik Spider,attack-pattern--65f2d882-3f41-4d48-8a06-29af77...,LSASS Memory,[credential-access],Adversaries may attempt to access credential m...,[Indrik Spider](https://attack.mitre.org/group...
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa...,LuminousMoth,attack-pattern--10ffac09-e42d-4f56-ab20-db94c6...,Steal Web Session Cookie,[credential-access],An adversary may steal web application or serv...,[LuminousMoth](https://attack.mitre.org/groups...
2,intrusion-set--3753cc21-2dae-4dfb-8481-d004e74...,FIN7,attack-pattern--ef67e13e-5598-4adc-bdb2-998225...,Malicious Link,[execution],An adversary may rely upon a user clicking a m...,[FIN7](https://attack.mitre.org/groups/G0046) ...
3,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--970a3432-3237-47ad-bcca-7d8cbb...,PowerShell,[execution],Adversaries may abuse PowerShell commands and ...,[WIRTE](https://attack.mitre.org/groups/G0090)...
4,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--b18eae87-b469-4e14-b454-b171b4...,Non-Standard Port,[command-and-control],Adversaries may communicate using a protocol a...,[WIRTE](https://attack.mitre.org/groups/G0090)...
5,intrusion-set--1c63d4ec-0a75-4daa-b1df-0d11af3...,Dragonfly,attack-pattern--53ac20cd-aca3-406e-9aa0-9fc7fd...,Archive Collected Data,[collection],An adversary may compress and/or encrypt data ...,[Dragonfly](https://attack.mitre.org/groups/G0...
6,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,attack-pattern--e7cbc1de-1f79-48ee-abfd-da1241...,Code Signing Certificates,[resource-development],Adversaries may buy and/or steal code signing ...,[Wizard Spider](https://attack.mitre.org/group...
7,intrusion-set--96e239be-ad99-49eb-b127-3007b8c...,Equation,attack-pattern--dfebc3b7-d19d-450b-81c7-6dafe4...,Hidden File System,[defense-evasion],Adversaries may use a hidden file system to co...,[Equation](https://attack.mitre.org/groups/G00...
8,intrusion-set--c21dd6f1-1364-4a70-a1f7-783080e...,Fox Kitten,attack-pattern--e6919abc-99f9-4c6c-95a5-14761e...,Ingress Tool Transfer,[command-and-control],Adversaries may transfer tools or other files ...,[Fox Kitten](https://attack.mitre.org/groups/G...
9,intrusion-set--c21dd6f1-1364-4a70-a1f7-783080e...,Fox Kitten,attack-pattern--7385dfaf-6886-4229-9ecd-6fd678...,Command and Scripting Interpreter,[execution],Adversaries may abuse command and script inter...,[Fox Kitten](https://attack.mitre.org/groups/G...


Time Period: 7, Number of Records: 3459


,group_id,group_name,technique_id,technique_name,tactic,technique_description,relationship_description
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d140...,Indrik Spider,attack-pattern--65f2d882-3f41-4d48-8a06-29af77...,LSASS Memory,[credential-access],Adversaries may attempt to access credential m...,[Indrik Spider](https://attack.mitre.org/group...
1,intrusion-set--3753cc21-2dae-4dfb-8481-d004e74...,FIN7,attack-pattern--ef67e13e-5598-4adc-bdb2-998225...,Malicious Link,[execution],An adversary may rely upon a user clicking a m...,[FIN7](https://attack.mitre.org/groups/G0046) ...
2,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--970a3432-3237-47ad-bcca-7d8cbb...,PowerShell,[execution],Adversaries may abuse PowerShell commands and ...,[WIRTE](https://attack.mitre.org/groups/G0090)...
3,intrusion-set--f8cb7b36-62ef-4488-8a6d-a7033e3...,WIRTE,attack-pattern--b18eae87-b469-4e14-b454-b171b4...,Non-Standard Port,[command-and-control],Adversaries may communicate using a protocol a...,[WIRTE](https://attack.mitre.org/groups/G0090)...
4,intrusion-set--1c63d4ec-0a75-4daa-b1df-0d11af3...,Dragonfly,attack-pattern--53ac20cd-aca3-406e-9aa0-9fc7fd...,Archive Collected Data,[collection],An adversary may compress and/or encrypt data ...,[Dragonfly](https://attack.mitre.org/groups/G0...
5,intrusion-set--dd2d9ca6-505b-4860-a604-233685b...,Wizard Spider,attack-pattern--e7cbc1de-1f79-48ee-abfd-da1241...,Code Signing Certificates,[resource-development],Adversaries may buy and/or steal code signing ...,[Wizard Spider](https://attack.mitre.org/group...
6,intrusion-set--96e239be-ad99-49eb-b127-3007b8c...,Equation,attack-pattern--dfebc3b7-d19d-450b-81c7-6dafe4...,Hidden File System,[defense-evasion],Adversaries may use a hidden file system to co...,[Equation](https://attack.mitre.org/groups/G00...
7,intrusion-set--c21dd6f1-1364-4a70-a1f7-783080e...,Fox Kitten,attack-pattern--e6919abc-99f9-4c6c-95a5-14761e...,Ingress Tool Transfer,[command-and-control],Adversaries may transfer tools or other files ...,[Fox Kitten](https://attack.mitre.org/groups/G...
8,intrusion-set--c21dd6f1-1364-4a70-a1f7-783080e...,Fox Kitten,attack-pattern--7385dfaf-6886-4229-9ecd-6fd678...,Command and Scripting Interpreter,[execution],Adversaries may abuse command and script inter...,[Fox Kitten](https://attack.mitre.org/groups/G...
9,intrusion-set--64b52e7d-b2c4-4a02-9372-08a463f...,Aquatic Panda,attack-pattern--b3d682b6-98f2-4fb0-aa3b-b4df00...,Obfuscated Files or Information,[defense-evasion],Adversaries may attempt to make an executable ...,[Aquatic Panda](https://attack.mitre.org/group...


In [6]:
import pandas as pd

# Given: dfs = {1: df1, 2: df2, 3: df3} where keys are time periods (1=newest, 3=oldest)

# ------------------------------------------------------------
# STEP 1: Compute usage per technique per period
# ------------------------------------------------------------
usage_frames = []

for period, df in dfs.items():
    total_groups = df["group_name"].nunique()
    
    usage = (
        df.groupby("technique_name")["group_name"]
        .nunique()
        .reset_index(name="num_groups_using")
    )
    
    usage["usage"] = usage["num_groups_using"] / total_groups
    usage["time_period"] = period
    
    usage_frames.append(usage[["technique_name", "time_period", "usage"]])

usage_df = pd.concat(usage_frames, ignore_index=True)

# ------------------------------------------------------------
# STEP 2: Pivot to wide format
# ------------------------------------------------------------
usage_wide = usage_df.pivot(
    index="technique_name",
    columns="time_period", 
    values="usage"
)

# Rename columns to be descriptive
# period 1 = current, period 2 = t-1, period 3 = t-2
column_mapping = {
    1: "usage_current",
    2: "usage_t_minus_1",
    3: "usage_t_minus_2",
    4: "usage_t_minus_3",
    5: "usage_t_minus_4",
    6: "usage_t_minus_5",
    7: "usage_t_minus_6",
}
usage_wide.columns = [column_mapping[col] for col in usage_wide.columns]

# Reset index to make technique_name a column
usage_wide = usage_wide.reset_index()

# Fill NaN with 0 (technique didn't exist in that period)
usage_wide = usage_wide.fillna(0)

# Result is now a dataframe
df = usage_wide
df.head(50)

,technique_name,usage_current,usage_t_minus_1,usage_t_minus_2,usage_t_minus_3,usage_t_minus_4,usage_t_minus_5,usage_t_minus_6
0,ARP Cache Poisoning,0.011905,0.012346,0.011905,0.012739,0.013245,0.013699,0.006993
1,Abuse Elevation Control Mechanism,0.005952,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Access Token Manipulation,0.017857,0.018519,0.011905,0.012739,0.013245,0.013699,0.013986
3,Accessibility Features,0.035714,0.037037,0.035714,0.038217,0.039735,0.041096,0.041958
4,Account Access Removal,0.011905,0.012346,0.011905,0.012739,0.006623,0.006849,0.006993
5,Account Discovery,0.017857,0.012346,0.029762,0.025478,0.026490,0.020548,0.020979
6,Account Manipulation,0.017857,0.012346,0.029762,0.070064,0.066225,0.061644,0.048951
7,Acquire Access,0.005952,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,Acquire Infrastructure,0.047619,0.043210,0.035714,0.006369,0.000000,0.000000,0.000000
9,Add-ins,0.005952,0.006173,0.005952,0.006369,0.006623,0.006849,0.006993


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Use all lag features:
lag_columns = [col for col in usage_wide.columns if col.startswith("usage_t_minus_")]
X = usage_wide[lag_columns]

# Target stays the same
y = usage_wide["usage_current"]

# Split into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# ------------------------------------------------------------
# STEP 2: Train multiple models
# ------------------------------------------------------------
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}")
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Evaluate
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    print(f"Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
    print(f"Train MAE:  {train_mae:.4f} | Test MAE:  {test_mae:.4f}")
    print(f"Train R²:   {train_r2:.4f} | Test R²:   {test_r2:.4f}")
    
    results.append({
        "Model": name,
        "Train RMSE": train_rmse,
        "Test RMSE": test_rmse,
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "Train R²": train_r2,
        "Test R²": test_r2
    })

# ------------------------------------------------------------
# STEP 3: Compare all models
# ------------------------------------------------------------
results_df = pd.DataFrame(results)
print(f"\n{'='*60}")
print("MODEL COMPARISON")
print(f"{'='*60}")
print(results_df.to_string(index=False))

# Find best model based on test RMSE
best_model_name = results_df.loc[results_df["Test RMSE"].idxmin(), "Model"]
print(f"\n🏆 Best Model: {best_model_name}")

# ------------------------------------------------------------
# STEP 4: Example predictions with best model
# ------------------------------------------------------------
best_model = models[best_model_name]

# Show some example predictions
example_indices = np.random.choice(len(X_test), size=10, replace=False)
examples = pd.DataFrame({
    "technique_name": usage_wide.iloc[X_test.index[example_indices]]["technique_name"].values,
    "usage_t_minus_2": X_test.iloc[example_indices]["usage_t_minus_2"].values,
    "usage_t_minus_1": X_test.iloc[example_indices]["usage_t_minus_1"].values,
    "actual_current": y_test.iloc[example_indices].values,
    "predicted_current": best_model.predict(X_test.iloc[example_indices])
})

print(f"\n{'='*60}")
print("EXAMPLE PREDICTIONS")
print(f"{'='*60}")
print(examples.to_string(index=False))

Training set size: 401
Test set size: 101

Training Linear Regression...
Train RMSE: 0.0060 | Test RMSE: 0.0045
Train MAE:  0.0042 | Test MAE:  0.0036
Train R²:   0.9946 | Test R²:   0.9963

Training Ridge Regression...
Train RMSE: 0.0196 | Test RMSE: 0.0270
Train MAE:  0.0101 | Test MAE:  0.0127
Train R²:   0.9430 | Test R²:   0.8684

Training Random Forest...
Train RMSE: 0.0049 | Test RMSE: 0.0069
Train MAE:  0.0024 | Test MAE:  0.0042
Train R²:   0.9964 | Test R²:   0.9915

Training Gradient Boosting...
Train RMSE: 0.0043 | Test RMSE: 0.0067
Train MAE:  0.0026 | Test MAE:  0.0042
Train R²:   0.9973 | Test R²:   0.9918

MODEL COMPARISON
            Model  Train RMSE  Test RMSE  Train MAE  Test MAE  Train R²  Test R²
Linear Regression    0.006020   0.004506   0.004180  0.003607  0.994618 0.996334
 Ridge Regression    0.019590   0.026994   0.010068  0.012663  0.943002 0.868432
    Random Forest    0.004901   0.006851   0.002446  0.004175  0.996432 0.991525
Gradient Boosting    0.004297

In [8]:
# ------------------------------------------------------------
# STEP 1: Train model using t-1 through t-6 to predict current
# ------------------------------------------------------------
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Features: t-1 through t-6
X = usage_wide[["usage_t_minus_1", "usage_t_minus_2", "usage_t_minus_3", 
                "usage_t_minus_4", "usage_t_minus_5", "usage_t_minus_6"]]
y = usage_wide["usage_current"]

# Split and train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"Test R²: {r2_score(y_test, y_pred):.4f}")

# ------------------------------------------------------------
# STEP 2: Shift data to predict NEXT period
# ------------------------------------------------------------
# Create a new dataframe where we shift all columns by 1
# What is "current" now becomes "t-1" for prediction purposes
X_future = usage_wide[["usage_current", "usage_t_minus_1", "usage_t_minus_2", 
                       "usage_t_minus_3", "usage_t_minus_4", "usage_t_minus_5"]].copy()

# Rename to match training feature names
X_future.columns = ["usage_t_minus_1", "usage_t_minus_2", "usage_t_minus_3", 
                    "usage_t_minus_4", "usage_t_minus_5", "usage_t_minus_6"]

# ------------------------------------------------------------
# STEP 3: Predict next time period
# ------------------------------------------------------------
predictions_next_period = model.predict(X_future)

# ------------------------------------------------------------
# STEP 4: Create results dataframe
# ------------------------------------------------------------
results = pd.DataFrame({
    "technique_name": usage_wide["technique_name"],
    "usage_current": usage_wide["usage_current"],
    "predicted_next_period": predictions_next_period,
    "predicted_change": predictions_next_period - usage_wide["usage_current"].values
})

# Sort by predicted usage
results = results.sort_values("predicted_next_period", ascending=False)

print("\nTop 20 techniques predicted for next period:")
print(results.head(20))

print("\nTechniques predicted to decline most:")
print(results.sort_values("predicted_change").head(20))

print("\nTechniques predicted to rise most:")
print(results.sort_values("predicted_change", ascending=False).head(20))
results.head(50)

Test RMSE: 0.0045
Test R²: 0.9963

Top 20 techniques predicted for next period:
                                 technique_name  usage_current  \
246                       Ingress Tool Transfer       0.505952   
285                              Malicious File       0.500000   
350                                  PowerShell       0.494048   
457                                        Tool       0.470238   
423                    Spearphishing Attachment       0.458333   
490                       Windows Command Shell       0.422619   
297  Match Legitimate Resource Name or Location       0.351190   
483                               Web Protocols       0.333333   
441                System Information Discovery       0.327381   
370          Registry Run Keys / Startup Folder       0.327381   
396                              Scheduled Task       0.321429   
209                File and Directory Discovery       0.297619   
424                          Spearphishing Link       0.297619

,technique_name,usage_current,predicted_next_period,predicted_change
246,Ingress Tool Transfer,0.505952,0.500373,-0.005579
285,Malicious File,0.500000,0.493174,-0.006826
350,PowerShell,0.494048,0.492778,-0.001270
457,Tool,0.470238,0.468727,-0.001511
423,Spearphishing Attachment,0.458333,0.450376,-0.007957
490,Windows Command Shell,0.422619,0.418249,-0.004370
297,Match Legitimate Resource Name or Location,0.351190,0.387474,0.036283
483,Web Protocols,0.333333,0.330061,-0.003272
441,System Information Discovery,0.327381,0.329579,0.002198
370,Registry Run Keys / Startup Folder,0.327381,0.323282,-0.004099


In [9]:
# ------------------------------------------------------------
# STEP 2: Iterative prediction function
# ------------------------------------------------------------
def predict_future_periods(usage_wide, model, num_periods=5):
    """
    Predict multiple future periods iteratively.
    
    Parameters:
    - usage_wide: DataFrame with current and historical usage
    - model: Trained model
    - num_periods: Number of future periods to predict
    
    Returns:
    - DataFrame with predictions for each future period
    """
    # Start with current data
    current_data = usage_wide.copy()
    
    # Store predictions
    predictions = {
        "technique_name": current_data["technique_name"],
        "usage_current": current_data["usage_current"]
    }
    
    # Keep track of the latest 6 periods (rolling window)
    latest_usage = current_data[[
        "usage_current", "usage_t_minus_1", "usage_t_minus_2",
        "usage_t_minus_3", "usage_t_minus_4", "usage_t_minus_5"
    ]].values
    
    for period in range(1, num_periods + 1):
        # Prepare features for prediction (rename to match training)
        X_future = pd.DataFrame(
            latest_usage,
            columns=["usage_t_minus_1", "usage_t_minus_2", "usage_t_minus_3",
                    "usage_t_minus_4", "usage_t_minus_5", "usage_t_minus_6"]
        )
        
        # Predict next period
        next_period_pred = model.predict(X_future)
        
        # Store prediction
        predictions[f"predicted_t_plus_{period}"] = next_period_pred
        
        # Shift the window: new prediction becomes most recent
        # Drop oldest (last column), shift everything right, add new prediction at front
        latest_usage = np.column_stack([next_period_pred, latest_usage[:, :-1]])
    
    return pd.DataFrame(predictions)

# ------------------------------------------------------------
# STEP 3: Predict next 5 periods
# ------------------------------------------------------------
future_predictions = predict_future_periods(usage_wide, model, num_periods=5)

# ------------------------------------------------------------
# STEP 4: Add delta columns (change from previous period)
# ------------------------------------------------------------
# Delta from current to t+1
future_predictions["delta_current_to_t_plus_1"] = (
    future_predictions["predicted_t_plus_1"] - future_predictions["usage_current"]
)

# Deltas between consecutive predictions
for i in range(1, 5):
    future_predictions[f"delta_t_plus_{i}_to_t_plus_{i+1}"] = (
        future_predictions[f"predicted_t_plus_{i+1}"] - future_predictions[f"predicted_t_plus_{i}"]
    )

# ------------------------------------------------------------
# STEP 5: Sort by predicted_t_plus_5 (highest to lowest)
# ------------------------------------------------------------
future_predictions = future_predictions.sort_values("predicted_t_plus_5", ascending=False)

print("Predictions for next 5 periods (sorted by t+5):")
print(future_predictions.head(20))

# Optional: Show specific columns for easier reading
display_cols = [
    "technique_name",
    "usage_current",
    "predicted_t_plus_1", "delta_current_to_t_plus_1",
    "predicted_t_plus_2", "delta_t_plus_1_to_t_plus_2",
    "predicted_t_plus_3", "delta_t_plus_2_to_t_plus_3",
    "predicted_t_plus_4", "delta_t_plus_3_to_t_plus_4",
    "predicted_t_plus_5", "delta_t_plus_4_to_t_plus_5"
]

print("\nWith deltas (top 20):")
future_predictions[display_cols].head(20)

Predictions for next 5 periods (sorted by t+5):
                                 technique_name  usage_current  \
297  Match Legitimate Resource Name or Location       0.351190   
350                                  PowerShell       0.494048   
246                       Ingress Tool Transfer       0.505952   
285                              Malicious File       0.500000   
457                                        Tool       0.470238   
423                    Spearphishing Attachment       0.458333   
490                       Windows Command Shell       0.422619   
483                               Web Protocols       0.333333   
441                System Information Discovery       0.327381   
396                              Scheduled Task       0.321429   
370          Registry Run Keys / Startup Folder       0.327381   
180                      Encrypted/Encoded File       0.220238   
209                File and Directory Discovery       0.297619   
424                         

,technique_name,usage_current,predicted_t_plus_1,delta_current_to_t_plus_1,predicted_t_plus_2,delta_t_plus_1_to_t_plus_2,predicted_t_plus_3,delta_t_plus_2_to_t_plus_3,predicted_t_plus_4,delta_t_plus_3_to_t_plus_4,predicted_t_plus_5,delta_t_plus_4_to_t_plus_5
297,Match Legitimate Resource Name or Location,0.351190,0.387474,0.036283,0.382337,-0.005137,0.471108,0.088771,0.492197,0.021089,0.505770,0.013573
350,PowerShell,0.494048,0.492778,-0.001270,0.491777,-0.001001,0.490529,-0.001248,0.489651,-0.000878,0.486998,-0.002653
246,Ingress Tool Transfer,0.505952,0.500373,-0.005579,0.496764,-0.003610,0.490852,-0.005911,0.485412,-0.005440,0.480372,-0.005040
285,Malicious File,0.500000,0.493174,-0.006826,0.485644,-0.007530,0.481692,-0.003952,0.475445,-0.006247,0.469966,-0.005479
457,Tool,0.470238,0.468727,-0.001511,0.464709,-0.004018,0.467013,0.002304,0.467562,0.000549,0.465496,-0.002066
423,Spearphishing Attachment,0.458333,0.450376,-0.007957,0.446325,-0.004051,0.439165,-0.007160,0.431138,-0.008026,0.425158,-0.005981
490,Windows Command Shell,0.422619,0.418249,-0.004370,0.420070,0.001821,0.412247,-0.007823,0.409178,-0.003069,0.405672,-0.003506
483,Web Protocols,0.333333,0.330061,-0.003272,0.327873,-0.002188,0.329566,0.001693,0.327687,-0.001880,0.325517,-0.002170
441,System Information Discovery,0.327381,0.329579,0.002198,0.332235,0.002656,0.329561,-0.002674,0.326375,-0.003187,0.325091,-0.001283
396,Scheduled Task,0.321429,0.317813,-0.003616,0.320007,0.002194,0.316804,-0.003203,0.315216,-0.001588,0.313091,-0.002124


In [16]:
future_predictions[['technique_name', 'predicted_t_plus_1', 'predicted_t_plus_2', 'predicted_t_plus_3', 'predicted_t_plus_4', 'predicted_t_plus_5']].head(20)

,technique_name,predicted_t_plus_1,predicted_t_plus_2,predicted_t_plus_3,predicted_t_plus_4,predicted_t_plus_5
297,Match Legitimate Resource Name or Location,0.387474,0.382337,0.471108,0.492197,0.505770
350,PowerShell,0.492778,0.491777,0.490529,0.489651,0.486998
246,Ingress Tool Transfer,0.500373,0.496764,0.490852,0.485412,0.480372
285,Malicious File,0.493174,0.485644,0.481692,0.475445,0.469966
457,Tool,0.468727,0.464709,0.467013,0.467562,0.465496
423,Spearphishing Attachment,0.450376,0.446325,0.439165,0.431138,0.425158
490,Windows Command Shell,0.418249,0.420070,0.412247,0.409178,0.405672
483,Web Protocols,0.330061,0.327873,0.329566,0.327687,0.325517
441,System Information Discovery,0.329579,0.332235,0.329561,0.326375,0.325091
396,Scheduled Task,0.317813,0.320007,0.316804,0.315216,0.313091


In [10]:
# Create simplified dataframe with just current, t+5, and delta
summary_df = pd.DataFrame({
    "technique_name": future_predictions["technique_name"],
    "usage_current": future_predictions["usage_current"],
    "predicted_t_plus_5": future_predictions["predicted_t_plus_5"],
    "delta_current_to_t_plus_5": future_predictions["predicted_t_plus_5"] - future_predictions["usage_current"]
})

# Sort by predicted_t_plus_5 (highest to lowest)
summary_df = summary_df.sort_values("predicted_t_plus_5", ascending=False)

print("Summary: Current vs Predicted t+5")
summary_df.head(20)

# Or display all rows if you want:
# print(summary_df)

Summary: Current vs Predicted t+5


,technique_name,usage_current,predicted_t_plus_5,delta_current_to_t_plus_5
297,Match Legitimate Resource Name or Location,0.351190,0.505770,0.154579
350,PowerShell,0.494048,0.486998,-0.007050
246,Ingress Tool Transfer,0.505952,0.480372,-0.025580
285,Malicious File,0.500000,0.469966,-0.030034
457,Tool,0.470238,0.465496,-0.004742
423,Spearphishing Attachment,0.458333,0.425158,-0.033176
490,Windows Command Shell,0.422619,0.405672,-0.016947
483,Web Protocols,0.333333,0.325517,-0.007816
441,System Information Discovery,0.327381,0.325091,-0.002290
396,Scheduled Task,0.321429,0.313091,-0.008337


In [11]:
summary_df.sort_values("delta_current_to_t_plus_5", ascending=False)

,technique_name,usage_current,predicted_t_plus_5,delta_current_to_t_plus_5
297,Match Legitimate Resource Name or Location,0.351190,0.505770,0.154579
109,DLL,0.190476,0.279699,0.089223
180,Encrypted/Encoded File,0.220238,0.306289,0.086051
373,Remote Access Tools,0.077381,0.120612,0.043231
212,Financial Theft,0.083333,0.118449,0.035116
...,...,...,...,...
423,Spearphishing Attachment,0.458333,0.425158,-0.033176
76,Commonly Used Port,0.000000,-0.047973,-0.047973
111,DLL Side-Loading,0.000000,-0.054411,-0.054411
328,Obfuscated Files or Information,0.107143,-0.001668,-0.108810


In [12]:
from sklearn.preprocessing import MinMaxScaler

# Create a scaler
scaler = MinMaxScaler()

# Apply min-max scaling to the delta column
summary_df["scaled_score"] = scaler.fit_transform(summary_df[["delta_current_to_t_plus_5"]])

# Sort by delta (highest to lowest)
summary_df = summary_df.sort_values("delta_current_to_t_plus_5", ascending=False)

print("Summary with scaled scores:")
summary_df.head(20)

Summary with scaled scores:


,technique_name,usage_current,predicted_t_plus_5,delta_current_to_t_plus_5,scaled_score
297,Match Legitimate Resource Name or Location,0.351190,0.505770,0.154579,1.000000
109,DLL,0.190476,0.279699,0.089223,0.776260
180,Encrypted/Encoded File,0.220238,0.306289,0.086051,0.765400
373,Remote Access Tools,0.077381,0.120612,0.043231,0.618813
212,Financial Theft,0.083333,0.118449,0.035116,0.591031
376,Remote Desktop Software,0.053571,0.085874,0.032303,0.581401
277,Local Storage Discovery,0.059524,0.091340,0.031816,0.579735
83,Compression,0.041667,0.069959,0.028292,0.567671
476,Virtual Private Server,0.095238,0.122365,0.027127,0.563681
241,Impersonation,0.047619,0.074107,0.026488,0.561495


In [13]:
# ------------------------------------------------------------
# STEP 1: Merge group techniques with scaled scores
# ------------------------------------------------------------
group_scores = group_techniques_df.merge(
    summary_df[["technique_name", "scaled_score"]],
    on="technique_name",
    how="left"
)

# Fill any missing scores with 0 (in case a technique doesn't have a score)
group_scores["scaled_score"] = group_scores["scaled_score"].fillna(0)

# ------------------------------------------------------------
# STEP 2: Sum scaled scores per group
# ------------------------------------------------------------
group_weighted_scores = (
    group_scores.groupby("group_name")["scaled_score"]
    .sum()
    .reset_index(name="total_weighted_score")
)

# Sort by total weighted score (highest to lowest)
group_weighted_scores = group_weighted_scores.sort_values("total_weighted_score", ascending=False)

print("Group Weighted Scores (sorted by total score):")
print(group_weighted_scores.head(20))

# Optional: Add count of techniques per group for additional context
technique_counts = (
    group_techniques_df.groupby("group_name")
    .size()
    .reset_index(name="num_techniques")
)

group_weighted_scores = group_weighted_scores.merge(technique_counts, on="group_name")

print("\nWith technique counts:")
group_weighted_scores.head(20)

Group Weighted Scores (sorted by total score):
           group_name  total_weighted_score
80            Kimsuky             54.288133
82      Lazarus Group             46.613825
7               APT28             45.391824
100     Mustang Panda             42.154619
156      Volt Typhoon             40.915974
16              APT41             40.724450
89        Magic Hound             39.560615
121     Sandworm Team             39.149036
11              APT32             38.917869
104            OilRig             37.577241
69    Gamaredon Group             34.286484
152             Turla             33.940033
61               FIN7             33.286046
8               APT29             32.986449
123  Scattered Spider             32.658376
164     Wizard Spider             31.059599
36            Chimera             29.703667
143           TeamTNT             28.610885
99         MuddyWater             28.589048
91       Medusa Group             28.504248

With technique counts:


,group_name,total_weighted_score,num_techniques
0,Kimsuky,54.288133,109
1,Lazarus Group,46.613825,93
2,APT28,45.391824,91
3,Mustang Panda,42.154619,85
4,Volt Typhoon,40.915974,81
5,APT41,40.724450,82
6,Magic Hound,39.560615,79
7,Sandworm Team,39.149036,79
8,APT32,38.917869,78
9,OilRig,37.577241,76


In [14]:
# Create a scaler
scaler = MinMaxScaler()

# Apply min-max scaling to the total weighted score
group_weighted_scores["scaled_group_score"] = scaler.fit_transform(
    group_weighted_scores[["total_weighted_score"]]
)

# Sort by scaled score (highest to lowest)
group_weighted_scores = group_weighted_scores.sort_values("scaled_group_score", ascending=False)

print("Group Weighted Scores with Min-Max Scaling:")
group_weighted_scores.head(20)

Group Weighted Scores with Min-Max Scaling:


,group_name,total_weighted_score,num_techniques,scaled_group_score
0,Kimsuky,54.288133,109,1.000000
1,Lazarus Group,46.613825,93,0.858381
2,APT28,45.391824,91,0.835831
3,Mustang Panda,42.154619,85,0.776092
4,Volt Typhoon,40.915974,81,0.753235
5,APT41,40.724450,82,0.749701
6,Magic Hound,39.560615,79,0.728223
7,Sandworm Team,39.149036,79,0.720628
8,APT32,38.917869,78,0.716362
9,OilRig,37.577241,76,0.691623


CSV exporting

In [15]:
# csv_df = group_weighted_scores[['group_name', 'scaled_group_score']]
# csv_df.rename(columns={'scaled_group_score': 'score'}, inplace=True)
# csv_df.to_csv("../analysis_data/technique_predictions.csv", index=False)